# 05. MODEL TRAINING, TUNING & EMPIRICAL BENCHMARKING
**Huấn luyện 4 mô hình học máy: Logistic Regression, Random Forest, XGBoost và LightGBM**
---
Nội dung notebook này thực hiện:
1. Huấn luyện 4 thuật toán phân loại có xử lý mất cân bằng lớp (`scale_pos_weight` / `class_weight='balanced'`).
2. Kiểm định chéo 5-Fold Stratified Cross-Validation.
3. Đánh giá toàn diện trên tập kiểm tra: Accuracy, Precision, Recall, F1-Score, ROC-AUC, PR-AUC, Confusion Matrix.
4. So sánh thực nghiệm với phương pháp lấy mẫu SMOTE.


In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix
import xgboost as xgb
import lightgbm as lgb

X_train_scaled = joblib.load('../dataset/processed/X_train_scaled.pkl')
X_test_scaled = joblib.load('../dataset/processed/X_test_scaled.pkl')
y_train = joblib.load('../dataset/processed/y_train.pkl')
y_test = joblib.load('../dataset/processed/y_test.pkl')

scale_pos_wt = float((y_train == 0).sum() / (y_train == 1).sum())

models = {
    'Logistic Regression': LogisticRegression(C=0.1, max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_split=10, random_state=42, class_weight='balanced', n_jobs=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.08, scale_pos_weight=scale_pos_wt, random_state=42, eval_metric='logloss', n_jobs=-1),
    'LightGBM': lgb.LGBMClassifier(n_estimators=150, max_depth=6, num_leaves=31, learning_rate=0.08, scale_pos_weight=scale_pos_wt, random_state=42, verbose=-1, n_jobs=-1)
}

for name, m in models.items():
    m.fit(X_train_scaled, y_train)
    y_prob = m.predict_proba(X_test_scaled)[:, 1]
    y_pred = m.predict(X_test_scaled)
    auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    print(f'=== {name} ===')
    print(f'ROC-AUC: {auc:.4f} | PR-AUC: {pr_auc:.4f}')
    print(classification_report(y_test, y_pred, digits=4))
